# Availability inference

Use the notebook-independent availability API to load the saved champion and predict a session's duration, expected free time, and remaining wait. The feature and inference implementation lives in src/availability, so the production handoff does not depend on this notebook.

The duration model estimates total duration from session start. It does not condition on elapsed charging time. Provide a causal history snapshot for best results; omitted numeric history uses training medians.

In [4]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "team" / "mariam_availability" / "src" / "availability" / "inference.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate ChargeSentinel-Twin repository root")
TEAM_DIR = ROOT / "team" / "mariam_availability"
sys.path.insert(0, str(TEAM_DIR))

from src.availability.inference import AvailabilityPredictor, predict_availability

predictor = AvailabilityPredictor()
print("Champion:", predictor.model_version)

Champion: xgboost_default@e518e9c6d140


## Example payload

The example uses categories from the saved encoder. Replace its demonstration timestamps and history with current values from charger state. Timestamps must use consistent local time because the dataset timezone is unresolved.

In [5]:
example_payload = {
    "charger_id": next(iter(predictor.encoder.maps["charger_id"])),
    "location": next(iter(predictor.encoder.maps["location"])),
    "district": next(iter(predictor.encoder.maps["district"])),
    "session_start": "2021-10-15T12:00:00",
    "as_of": "2021-10-15T12:30:00",
    "history_features": {
        "hist_abnormal_rate": 0.12,
        "sessions_before": 20,
        "has_history": 1,
    },
}

# Reuse the loaded predictor for repeated serving calls.
prediction = predictor.predict(example_payload)
prediction

{'charger_id': '1',
 'predicted_duration_minutes': 38.57,
 'expected_free_time': '2021-10-15T12:38:33.967375452',
 'expected_wait_minutes': 8.57,
 'anomaly_risk': 0.12,
 'model_version': 'xgboost_default@e518e9c6d140'}

## Output and limits

The response returns predicted duration, expected free time, remaining wait, historical anomaly risk (when supplied), and model version. It rejects end time, duration, payment, energy, cost, and outcome flags. For already-running sessions, expected free time is based on total duration predicted from the original start; it is not a survival-adjusted remaining-time estimate.